In [3]:
# setup
import anthropic
from dotenv import load_dotenv
from anthropic import Anthropic, APIError

from __future__ import annotations

import json
import os
import sys
from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Callable

from anthropic import Anthropic, APIError

# Load variables from the .env file into the system environment
load_dotenv()

# Retrieve the variables using os.getenv()
api_key = os.getenv("API_SECRET_KEY")
# print(f"API Key: {api_key}")

MODEL_NAME = "claude-sonnet-4-5"
MAX_ITERATIONS = 8                        # hard ceiling on model<->tool round trips
MAX_TOOL_RETRIES_PER_CALL = 1             # a single tool failing this many times kills the loop
AUTONOMOUS_REFUND_LIMIT_USD = 50.00       # policy boundary, enforced in code, not by the model



In [4]:
# --------------------------------------------------------------------------
# Fake backend data (stand-ins for real account/order services)
# --------------------------------------------------------------------------

_ACCOUNTS_DB = {
    "acct_101": {"account_id": "acct_101", "name": "Dana Whitfield", "tier": "standard", "flagged": False},
    "acct_202": {"account_id": "acct_202", "name": "Marcus Ito", "tier": "premium", "flagged": False},
    "acct_303": {"account_id": "acct_303", "name": "Priya Shah", "tier": "standard", "flagged": True},
}

_ORDERS_DB = {
    "ord_9001": {"order_id": "ord_9001", "account_id": "acct_101", "amount_usd": 19.99, "status": "delivered", "issue": "item_missing"},
    "ord_9002": {"order_id": "ord_9002", "account_id": "acct_202", "amount_usd": 249.00, "status": "delivered", "issue": "not_as_described"},
    "ord_9003": {"order_id": "ord_9003", "account_id": "acct_303", "amount_usd": 34.50, "status": "delivered", "issue": "duplicate_charge"},
}


In [5]:
# classes
class Outcome(str, Enum):
    RESOLVED_AUTONOMOUSLY = "RESOLVED_AUTONOMOUSLY"
    ESCALATED = "ESCALATED"
    FAILED_TOOL_ERROR = "FAILED_TOOL_ERROR"
    MAX_ITERATIONS_HIT = "MAX_ITERATIONS_HIT"

# --------------------------------------------------------------------------
# Tool implementations
#
# Each tool raises a ToolError for expected failure modes, which the loop
# converts into a structured, model-visible error rather than a crash.
# --------------------------------------------------------------------------
 
class ToolError(Exception):
    """Raised by tool implementations for expected, recoverable failures."""
 
 
def tool_get_account(account_id: str) -> dict:
    account = _ACCOUNTS_DB.get(account_id)
    if account is None:
        raise ToolError(f"No account found with id '{account_id}'")
    return account
 
 
def tool_get_order(order_id: str) -> dict:
    order = _ORDERS_DB.get(order_id)
    if order is None:
        raise ToolError(f"No order found with id '{order_id}'")
    return order
 
 
def tool_issue_refund(order_id: str, amount_usd: float, reason: str) -> dict:
    order = _ORDERS_DB.get(order_id)
    if order is None:
        raise ToolError(f"Cannot refund unknown order '{order_id}'")
    if amount_usd <= 0:
        raise ToolError("Refund amount must be positive")
    if amount_usd > order["amount_usd"]:
        raise ToolError(
            f"Refund amount {amount_usd} exceeds order total {order['amount_usd']}"
        )
    # NOTE: this function is never reachable for over-threshold refunds --
    # the loop intercepts those before dispatch. See AgentLoop._dispatch_tool.
    return {
        "status": "refund_issued",
        "order_id": order_id,
        "amount_usd": amount_usd,
        "reason": reason,
    }
 
 
def tool_escalate_to_human(reason: str, recommended_action: str) -> dict:
    return {
        "status": "escalated",
        "reason": reason,
        "recommended_action": recommended_action,
    }
 
 
def tool_finish(summary: str) -> dict:
    return {"status": "finished", "summary": summary}
 
 
TOOL_IMPL: dict[str, Callable[..., dict]] = {
    "get_account": tool_get_account,
    "get_order": tool_get_order,
    "issue_refund": tool_issue_refund,
    "escalate_to_human": tool_escalate_to_human,
    "finish": tool_finish,
}


# account = tool_get_account("acct_101")
# print(f"Account: {account}")

In [6]:
# --------------------------------------------------------------------------
# Tool schemas: https://platform.claude.com/docs/en/agents-and-tools/tool-use/define-tools
# --------------------------------------------------------------------------

TOOLS = [
    {
        "name": "get_account",
        "description": "Fetch customer account information by account Id.",
        "input_schema":{
            "type": "object",
            "properties":{
                "account_id": {"type": "string", "description": "The unique identifier of the customer account."},
                "required": ["account_id"]
            },
        },
    }, 
    {
        "name": "get_order",
        "description": "Fetch order information by order Id.",
        "input_schema":{
            "type": "object",
            "properties" : { "order_id": {"type": "string"} },
            "required" : ["order_id"]
        }
    },
    {
        "name": "issue_refund",
        "description": (
            "Issue a refund for an order. HIGH RISK. Refunds above the "
            "autonomous limit will be automatically routed to a human "
            "instead of executed -- do not assume every call succeeds."
        ),
        "input_schema" : {
            "type": "object",
            "properties" : {
                "order_id" : {"type": "string", "description": "The unique identifier of the order to refund."},
                "amount_usd" : {"type": "number", "description": "The amount to refund in USD."},
                "reason" : {"type": "string", "description": "The reason for issuing the refund."}
            },
            "required" : ["order_id", "amount_usd", "reason"],
        }
    },
    {
        "name" : "escalate_to_human",
        "description": (
            "Hand this ticket off to a human agent instead of acting "
            "autonomously. Use for anything outside your authority "
            "(large refunds, account deletion, policy exceptions) or "
            "when you are not confident in the resolution."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "reason": {"type": "string"},
                "recommended_action": {"type": "string"},
            },
            "required": ["reason", "recommended_action"],
        },
    },
    {
        "name": "finish",
        "description": (
            "Call this exactly once you have fully resolved the ticket "
            "autonomously (no escalation needed) and are ready to end the "
            "loop. Provide a short summary of what was done."
        ),
        "input_schema": {
            "type": "object",
            "properties": {"summary": {"type": "string"}},
            "required": ["summary"],
        },
    },
]

SYSTEM_PROMPT = f"""You are an autonomous SaaS customer support agent.

You read a support ticket, use tools to look up the account and order, and
either resolve the issue yourself or hand it off to a human.

Hard rules, not suggestions:
- You may issue refunds up to ${AUTONOMOUS_REFUND_LIMIT_USD:.2f} autonomously.
  Any larger refund, and any account deletion or similarly irreversible
  account action, is OUTSIDE your authority. Call escalate_to_human instead
  of attempting it -- do not try to negotiate the amount down just to stay
  under the limit.
- When you have fully resolved the ticket yourself, call `finish` exactly
  once with a short summary. Do not call finish if you escalated instead.
- If a tool returns an error, do not immediately retry the same call with
  the same arguments. Either correct the input, try a different approach,
  or escalate.
- Never fabricate account or order data. Only use what tools return.
"""